# This Jupyter notebook demonstrates a Text-to-SQL Agent workflow.
### It showcases two approaches:
####   1. LangChain's Database Toolkit
####   2. LangChain's legacy SQL Agent with built-in tools
##### The examples highlight how natural language queries can be translated into SQL for database interactions.


In [1]:
### Download Database file

# import requests

# url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"

# response = requests.get(url)

# if response.status_code == 200:
#     # Open a local file in binary write mode
#     with open("Chinook.db", "wb") as file:
#         # Write the content of the response (the file) to the local file
#         file.write(response.content)
#     print("File downloaded and saved as Chinook.db")
# else:
#     print(f"Failed to download the file. Status code: {response.status_code}")

In [ ]:
from langchain_community.llms import Ollama

llm_model = Ollama(base_url = "http://localhost:11434", model = "gemma2:9b")

In [ ]:
# llm_model.invoke("who are you")

"I am Gemma, an open-weights AI assistant. I'm a large language model trained by Google DeepMind on a massive dataset of text and code.\n\nHere are some key things to know about me:\n\n* **Open-Weights:** My weights are publicly accessible, meaning anyone can see and use the underlying code that makes me work. This promotes transparency and collaboration in the AI community.\n* **Text-Based:** I operate solely through text. I can read your questions, understand them, and generate responses as text.\n* **No Real-World Interaction:** I don't have access to the internet or any real-world tools. I can't perform actions like searching the web, sending emails, or controlling devices.\n* **Knowledge Cut-Off:** My knowledge is based on the data I was trained on, which has a specific cut-off point. I don't have information about events that happened after my training.\n\nI'm here to help you with tasks like:\n\n* Answering your questions to the best of my ability based on my training data.\n* G

In [18]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import create_sql_agent
from langchain.agents.agent_types import AgentType

In [6]:
from langchain_community.utilities import SQLDatabase

database = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [7]:
print("Database: " + str(database.dialect))

Database: sqlite


In [9]:
print("Table Names in the Database: " + str(database.get_usable_table_names()))

Table Names in the Database: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [11]:
## Sample test

query = "Select * from Track LIMIT 5 "
run_sample = database.run(command= query)

print("Sample Output: ")
print(run_sample)

Sample Output: 
[(1, 'For Those About To Rock (We Salute You)', 1, 1, 1, 'Angus Young, Malcolm Young, Brian Johnson', 343719, 11170334, 0.99), (2, 'Balls to the Wall', 2, 2, 1, None, 342562, 5510424, 0.99), (3, 'Fast As a Shark', 3, 2, 1, 'F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman', 230619, 3990994, 0.99), (4, 'Restless and Wild', 3, 2, 1, 'F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. Dirkscneider & W. Hoffman', 252051, 4331779, 0.99), (5, 'Princess of the Dawn', 3, 2, 1, 'Deaffy & R.A. Smith-Diesel', 375418, 6290521, 0.99)]


In [13]:
toolkit = SQLDatabaseToolkit(db = database , llm = llm_model)

In [14]:
tools = toolkit.get_tools()

In [16]:
for tool in tools:
    print("Tool Name: " + str(tool.name))
    print("Tool Description: " + str(tool.description))
    print("\n")

Tool Name: sql_db_query
Tool Description: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.


Tool Name: sql_db_schema
Tool Description: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3


Tool Name: sql_db_list_tables
Tool Description: Input is an empty string, output is a comma-separated list of tables in the database.


Tool Name: sql_db_query_checker
Tool Description: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!




In [17]:
system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(dialect = database.dialect , top_k = 5)

In [20]:
agent = create_sql_agent(
    llm = llm_model,
    toolkit= toolkit,
    verbose= True,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION
)

In [1]:
# agent

In [24]:
query = "Which playlist has the most diverse genres?"
execute_query = agent.run(query)

/tmp/ipykernel_651189/3376789727.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  execute_query = agent.run(query)




> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, TrackThought: The `Playlist` and `Genre` tables seem most relevant. I should query their schemas.
Action: sql_db_schema
Action Input: Playlist, Genre
CREATE TABLE "Genre" (
	"GenreId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("GenreId")
)

/*
3 rows from Genre table:
GenreId	Name
1	Rock
2	Jazz
3	Metal
*/


CREATE TABLE "Playlist" (
	"PlaylistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("PlaylistId")
)

/*
3 rows from Playlist table:
PlaylistId	Name
1	Music
2	Movies
3	TV Shows
*/Thought: I need to figure out how the `Genre` and `Playlist` tables are connected.  The `PlaylistTrack` table likely connects playlists to tracks, and tracks have a genre association.

Action: sql_db_schema
Action Input: PlaylistTrack, Track
CREATE TABLE "PlaylistTrack" (
	"PlaylistId" INTEGER NOT N

In [25]:
print(execute_query)

Playlist with ID 8 has the most diverse genres with 20 different genres.
